# Lorentzian Model Definition

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from lmfit import minimize, Parameters, report_fit
import csv
import time
import warnings
from scipy.optimize import minimize as scipy_minimize


warnings.filterwarnings('ignore')

In [2]:
def lorentzian(x, x0, gamma, maximum):
    return maximum / (1.0 + ((x - x0) / gamma)**2)


def addWhiteNoiseRandom(yData, sigma):
    y_noisy = yData + np.random.normal(0, sigma, len(yData))
    return y_noisy


def addNoiseSNR(yData, SNR):
    signal_power = np.sum(yData**2) / len(yData)
    noise_power = np.sqrt(signal_power / SNR)
    y_noisy = yData + np.random.normal(0, noise_power, len(yData))
    return y_noisy, noise_power

In [3]:
def residual(params, x, data):
    x0 = params['x0']
    gamma = params['gamma']
    maximum = params['maximum']
    
    model = lorentzian(x, x0, gamma, maximum)
    return model - data

def lorentzian_grad(params, x, data):
    x0 = params['x0'].value if hasattr(params['x0'], 'value') else params['x0']
    gamma = params['gamma'].value if hasattr(params['gamma'], 'value') else params['gamma']
    maximum = params['maximum'].value if hasattr(params['maximum'], 'value') else params['maximum']
    
    model = lorentzian(x, x0, gamma, maximum)
    R = model - data
    
    u = (x - x0) / gamma
    denom = (1.0 + u**2)
    
    # Partial derivatives of the model w.r.t. parameters
    d_max = 1.0 / denom
    d_x0 = (2.0 * maximum * u) / (gamma * denom**2)
    d_gamma = (2.0 * maximum * u**2) / (gamma * denom**2)
    
    # Gradient of 0.5 * sum(R^2) -> J^T @ R
    grad_max = np.sum(R * d_max)
    grad_x0 = np.sum(R * d_x0)
    grad_gamma = np.sum(R * d_gamma)
    
    # Order matches build_params(): x0, gamma, maximum
    return np.array([grad_x0, grad_gamma, grad_max])

def lorentzian_hess(params, x, data):
    x0 = params['x0'].value if hasattr(params['x0'], 'value') else params['x0']
    gamma = params['gamma'].value if hasattr(params['gamma'], 'value') else params['gamma']
    maximum = params['maximum'].value if hasattr(params['maximum'], 'value') else params['maximum']
    
    u = (x - x0) / gamma
    denom = 1.0 + u**2
    model = lorentzian(x, x0, gamma, maximum)
    R = model - data
    
    # First derivatives of model
    d_x0 = (2.0 * maximum * u) / (gamma * denom**2)
    d_gamma = (2.0 * maximum * u**2) / (gamma * denom**2)
    d_max = 1.0 / denom
    
    # Second derivatives of model w.r.t parameters [x0, gamma, maximum]
    # d2_model / dx0^2
    d2_x0_x0 = (2.0 * maximum * (3.0 * u**2 - 1.0)) / (gamma**2 * denom**3)
    # d2_model / dx0_dgamma
    d2_x0_gamma = (2.0 * maximum * u * (3.0 * u**2 - 1.0)) / (gamma**2 * denom**3)
    # d2_model / dx0_dmax
    d2_x0_max = (2.0 * u) / (gamma * denom**2)
    
    # d2_model / dgamma^2
    d2_gamma_gamma = (2.0 * maximum * u**2 * (3.0 * u**2 - 2.0)) / (gamma**2 * denom**3)
    # d2_model / dgamma_dmax
    d2_gamma_max = (2.0 * u**2) / (gamma * denom**2)
    
    # d2_model / dmax^2
    d2_max_max = np.zeros_like(x)
    
    # Approximate Hessian of objective function: H ≈ J^T J + sum(R_i * D2_model_i)
    H = np.zeros((3, 3))
    
    # Parameter order: [x0, gamma, maximum]
    H[0, 0] = np.sum(d_x0**2 + R * d2_x0_x0)
    H[0, 1] = H[1, 0] = np.sum(d_x0 * d_gamma + R * d2_x0_gamma)
    H[0, 2] = H[2, 0] = np.sum(d_x0 * d_max + R * d2_x0_max)
    
    H[1, 1] = np.sum(d_gamma**2 + R * d2_gamma_gamma)
    H[1, 2] = H[2, 1] = np.sum(d_gamma * d_max + R * d2_gamma_max)
    
    H[2, 2] = np.sum(d_max**2 + R * d2_max_max)
    
    return H

# Experiment I: Fit Success Rate per Noise per Algorithm & ROC curves

# Error Thresholds

In [4]:
def center_pixel_error(true_val, fit_val, x_min, x_max, num_points, pixel_threshold):
    pixel_val = (x_max - x_min)/num_points
    deviation = abs(true_val - fit_val)
    pixel_deviation = deviation/pixel_val

    return pixel_deviation <= pixel_threshold
    

In [5]:
def relative_error(true_val, fit_val, threshold):
    rel_deviation = abs(true_val - fit_val)/true_val
    return rel_deviation <= threshold

## Running the Experiment

In [ ]:
# ---------------- Peak Params ----------------
num_points = 1000
x_min, x_max = -500.0, 500.0
center, width, amplitude = 0.0, 10.0, 1.0

# ---------------- Experiment Params ----------------
min_noise, max_noise = 0.0, 0.15
num_noise_groups = 50
noise_levels = np.linspace(min_noise, max_noise, num_noise_groups)
samples_per_group = 1000

x_data = np.linspace(x_min, x_max, num_points)
y_clean = lorentzian(x_data, center, width, amplitude)

# ---------------- Minimizer setup ----------------
# All methods lmfit supports. Some are local (fast), some global (slow),
# some gradient-based (benefit from jacobians), some derivative-free.


METHODS = [
    'leastsq', 'least_squares', 'nelder',
    'powell', 'cg', 'bfgs', 'cobyla',
    'lbfgsb', 'tnc', 'slsqp', 'trust-constr',
    'trust-ncg', 'trust-exact', 'trust-krylov', 'Newton-CG', 'dogleg'
]



# Per-method extra kwargs (only what's needed to make them practical).
# Global methods need sensible bounds / iteration budgets; otherwise they
# either fail outright or run for hours at 10k spectra * 23 methods.
BOUNDS = dict(x0=(-50, 50), gamma=(0.1, 100), maximum=(0.0, 5.0))

METHOD_KWARGS = {
    'leastsq':        {},
    'least_squares':  {},
    'nelder':         {},
    'lbfgsb':         {},
    'powell':         {},
    'cg':             {},
    'Newton-CG':      {},
    'cobyla':         {},
    'bfgs':           {},
    'tnc':            {},
    'trust-ncg':      {},
    'trust-exact':    {},
    'trust-krylov':   {},
    'trust-constr':   {},
    'dogleg':         {},
    'slsqp':          {},
}

def build_params():
    p = Parameters()
    p.add('x0',      value=center)
    p.add('gamma',   value=width,  min=0)
    p.add('maximum', value=amplitude)
    return p

def eval_one(method, y_noisy):
    params = build_params()
    kw = METHOD_KWARGS.get(method, {}).copy()

    start = time.perf_counter()
    try:
        # Special direct handling for dogleg and trust-exact via SciPy
        if method in ['dogleg', 'trust-exact']:
            # Define raw objective function for scipy (taking flat array x)
            def scipy_obj(x):
                p = build_params()
                p['x0'].value = x[0]
                p['gamma'].value = x[1]
                p['maximum'].value = x[2]
                return residual(p, x_data, y_noisy)

            # Define raw analytical Jacobian for scipy
            def scipy_jac(x):
                p = build_params()
                p['x0'].value = x[0]
                p['gamma'].value = x[1]
                p['maximum'].value = x[2]
                return lorentzian_grad(p, x_data, y_noisy)

            # Define raw analytical Hessian for scipy
            def scipy_hess(x):
                p = build_params()
                p['x0'].value = x[0]
                p['gamma'].value = x[1]
                p['maximum'].value = x[2]
                return lorentzian_hess(p, x_data, y_noisy)

            x0_init = np.array([center, width, amplitude])
            res = scipy_minimize(
                lambda x: 0.5 * np.sum(scipy_obj(x)**2), # SciPy scalar objective (0.5 * sum(R^2))
                x0_init,
                method=method,
                jac=scipy_jac,
                hess=scipy_hess
            )
            
            success = bool(res.success)
            nfe = res.nfev
            x0_f, gamma_f, amp_f = res.x[0], res.x[1], res.x[2]
            errmsg = ''
            
        else:
            # Standard lmfit path for all other methods
            if method in ['trust-ncg', 'trust-krylov', 'Newton-CG']:
                kw.update(
                    jac=lambda params, *args: lorentzian_grad(params, x_data, y_noisy),
                    hess='2-point'
                )
            
            result = minimize(residual, params, args=(x_data, y_noisy), method=method, **kw)
            success = bool(result.success)
            nfe = result.nfev
            x0_f    = result.params['x0'].value
            gamma_f = result.params['gamma'].value
            amp_f   = result.params['maximum'].value
            errmsg  = ''
            
    except Exception as e:
        success = False
        nfe = -1
        x0_f = gamma_f = amp_f = float('nan')
        errmsg = type(e).__name__ + ':' + str(e)[:80]
        
    runtime = (time.perf_counter() - start) * 1000
    return success, x0_f, gamma_f, amp_f, nfe, runtime, errmsg

# ---------------- Main loop ----------------
header = ['id', 'method', 'noise_gp', 'noise_val',
          'x0_true', 'gamma_true', 'amp_true',
          'x0_f', 'gamma_f', 'amp_f',
          'success', 'nfe', 'time (ms)', 'error']

total = len(METHODS) * len(noise_levels) * samples_per_group
index = 0
samples_for_image = []

with open('Experiment_All_Methods_Results.csv', 'w', newline='') as f:
    writer = csv.writer(f, delimiter=',')
    writer.writerow(header)

    for method in METHODS:
        noise_group_id = 0
        for noise in noise_levels:
            for i in range(samples_per_group):
                y_noisy = addWhiteNoiseRandom(y_clean, sigma=noise)
                success, x0_f, gamma_f, amp_f, nfe, runtime, err = \
                    eval_one(method, y_noisy)

                writer.writerow([index, method, noise_group_id, noise,
                                 center, width, amplitude,
                                 x0_f, gamma_f, amp_f,
                                 success, nfe, runtime, err])

                index += 1
                print(f"[{method}] {index}/{total} curves done",
                      end='\r')
            samples_for_image.append({'noise': noise, 'curve': y_noisy})
            noise_group_id += 1


[leastsq] 36476/800000 curves done

In [ ]:
n = len(samples_for_image[:10])
ncols = 5
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(3.0 * ncols, 2.8 * nrows),
    sharex=True, sharey=True,
    constrained_layout=True,
)
axes = np.atleast_2d(axes).ravel()

for ax, sample in zip(axes, samples_for_image):
    ax.plot(x_data, sample['curve'], lw=0.8)
    ax.set_title(f"$\\sigma = {sample['noise']:.2f}$", fontsize=10)
    ax.tick_params(labelsize=8)

# Hide unused panels
for ax in axes[n:]:
    ax.set_visible(False)

for ax in axes[:n]:
    ax.set_xlabel("cm$^{-1}$", fontsize=9)
for ax in axes[::ncols]:
    ax.set_ylabel("Intensity (a.u.)", fontsize=9)

fig.suptitle("Simulated Raman Samples", fontsize=12)
fig.savefig("simulated_samples_grid.png", dpi=150)
plt.show()
plt.close()

## Analysis of Results - Center

In [ ]:
n_noise = len(noise_levels)
n_methods = len(METHODS)

# Accumulate success counts per (method, noise_group)
center_succ = np.zeros((n_methods, n_noise))
# Also count actual rows seen per (method, noise_group) in case some were dropped
counts      = np.zeros((n_methods, n_noise), dtype=int)

method_idx = {m: i for i, m in enumerate(METHODS)}

with open('Experiment_All_Methods_Results.csv', 'r', newline='') as f:
    reader = csv.DictReader(f)          # header names drive column access
    for row in reader:
        m = row['method']
        if m not in method_idx:
            continue
        mi = method_idx[m]
        gi = int(row['noise_gp'])

        # Skip failed / nan fits — decide your policy here
        if row['success'].strip().lower() != 'true':
            counts[mi, gi] += 1          # still counts toward denominator
            continue

        center_t = np.float64(row['x0_true'])
        center_f = np.float64(row['x0_f'])

        if np.isnan(center_f):
            counts[mi, gi] += 1
            continue

        ok = center_pixel_error(center_t, center_f,
                                x_min, x_max, num_points,
                                pixel_threshold=1)
        center_succ[mi, gi] += ok
        counts[mi, gi]      += 1

# Normalize (guard against zero counts)
with np.errstate(invalid='ignore', divide='ignore'):
    prob = np.where(counts > 0, center_succ / counts * 100.0, np.nan)

# ---------------- Plot ----------------
fig, ax = plt.subplots(figsize=(9, 6))
cmap = plt.get_cmap('tab20')

for mi, m in enumerate(METHODS):
    ax.plot(noise_levels, prob[mi],
            marker='o', ms=3, lw=1.3,
            color=cmap(mi % 20),
            label=f"{m}")

ax.set_title("Center Fit Success Probability (all methods) for threshold = 1 pixel")
ax.set_xlabel(r"Noise $\sigma$")
ax.set_ylabel("Success probability (%)")
ax.set_ylim(91, 100.5)
ax.grid(alpha=0.3)
ax.legend(fontsize=7, ncol=2, loc='lower left', frameon=True)
fig.tight_layout()
fig.savefig("fit_success_all_methods.png", dpi=150)
plt.show()
plt.close()
        

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ---- Load results ----
df = pd.read_csv('Experiment_All_Methods_Results.csv')

# Coerce types defensively (some rows may have NaN / empty error strings)
df['time (ms)'] = pd.to_numeric(df['time (ms)'], errors='coerce')
df['noise_val'] = pd.to_numeric(df['noise_val'], errors='coerce')
df['success']   = df['success'].astype(str).str.lower().isin(['true', '1'])

# ---- Decide what counts toward runtime ----
# Option A: only successful fits (recommended — failures often short-circuit)
# Option B: all attempts (includes fast crashes from 'newton' etc.)
USE_SUCCESS_ONLY = True

work = df[df['success']] if USE_SUCCESS_ONLY else df.copy()
work = work.dropna(subset=['time (ms)', 'noise_val'])

# ---- Aggregate: mean runtime per (method, noise) ----
agg = (work.groupby(['method', 'noise_val'])['time (ms)']
            .agg(['mean', 'median', 'count'])
            .reset_index())

# ---- Plot ----
fig, ax = plt.subplots(figsize=(11, 7))

methods = sorted(agg['method'].unique())
cmap = plt.get_cmap('tab20')
colors = {m: cmap(i % 20) for i, m in enumerate(methods)}

for method in methods:
    sub = agg[agg['method'] == method].sort_values('noise_val')
    if sub.empty:
        continue
    ax.plot(sub['noise_val'], sub['mean'],
            marker='o', markersize=4, linewidth=1.5,
            label=method, color=colors[method])

ax.set_xlabel('Noise σ')
ax.set_ylabel('Average runtime (ms)')
title = 'Average fit runtime vs noise σ'
if USE_SUCCESS_ONLY:
    title += '  (successful fits only)'
ax.set_title(title)
ax.set_yscale('log')          # runtimes span orders of magnitude
ax.grid(True, which='both', alpha=0.3)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left',
          fontsize=9, ncol=1, frameon=False)
fig.tight_layout()
fig.savefig('avg_runtime_vs_noise.png', dpi=150, bbox_inches='tight')
plt.show()